# Step 9: Reproducible Data Split and Baseline Model

## 1. Executive Summary & Splitting Protocol
- **URL Deduplication**: 11 duplicate URL instances were removed to prevent split leakage, leaving **29,989 clean URL records**.
- **Stratified 80/20 Train/Test Split**: Executed with `random_state=42` and stratification on `is_phishing`.
- **Training Set (80%)**: **23,991 samples** (13,725 Legitimate, 10,266 Phishing)
- **Held-Out Test Set (20%)**: **5,998 samples** (3,432 Legitimate, 2,566 Phishing) — **LOCKED** until final Step 12 evaluation.
- **Cross-Validation Scheme**: 5-Fold Stratified Cross-Validation (`random_state=42`, `shuffle=True`) on training data.
- **Baseline Models Established**:
  1. Dummy Majority Class Classifier (`strategy='most_frequent'`)
  2. Simple Logistic Regression Baseline (`C=1.0`, 5-fold CV on train set)

In [1]:
import sys
import pandas as pd
from pathlib import Path

sys.path.append("..")
from src.train import prepare_and_split_data
from src.config import PROCESSED_DATA_DIR

# Execute reproducible data split and save processed datasets
X_train, X_test, y_train, y_test = prepare_and_split_data(save_splits=True)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape:  {X_test.shape}")
print(f"Train Phishing ratio: {y_train.mean()*100:.2f}%")
print(f"Test Phishing ratio:  {y_test.mean()*100:.2f}%")


Saved processed train split: C:\Users\JINIL\Desktop\Sem5_Innovative\Phish_Guard\data\processed\train.csv (23991 rows)
Saved processed test split: C:\Users\JINIL\Desktop\Sem5_Innovative\Phish_Guard\data\processed\test.csv (5998 rows)
Training set shape: (23991, 18)
Testing set shape:  (5998, 18)
Train Phishing ratio: 42.79%
Test Phishing ratio:  42.78%


## 2. Baseline Model 1: Dummy Majority Class Classifier
Evaluates naive majority class predictions (`is_phishing = 0`) to anchor performance.

In [2]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)
dummy_preds = dummy.predict(X_train)

print("=== DUMMY BASELINE PERFORMANCE (TRAIN SET) ===")
print(f"Accuracy:  {accuracy_score(y_train, dummy_preds):.4f}")
print(f"Precision: {precision_score(y_train, dummy_preds, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_train, dummy_preds, zero_division=0):.4f}")
print(f"F1-Score:  {f1_score(y_train, dummy_preds, zero_division=0):.4f}")


=== DUMMY BASELINE PERFORMANCE (TRAIN SET) ===
Accuracy:  0.5721
Precision: 0.0000
Recall:    0.0000
F1-Score:  0.0000


## 3. Baseline Model 2: Simple Logistic Regression (5-Fold CV on Train Set)
Evaluated strictly on training folds using `sklearn.pipeline.Pipeline` with `StandardScaler`.

In [3]:
from sklearn.linear_model import LogisticRegression
from src.feature_extraction import FS_URL_NUMERIC, FS_URL_BINARY
from src.preprocessing import get_feature_preprocessor, create_modeling_pipeline
from src.evaluate import evaluate_cv_pipeline

preprocessor = get_feature_preprocessor(FS_URL_NUMERIC, FS_URL_BINARY)
lr_model = LogisticRegression(random_state=42, max_iter=1000)
baseline_pipeline = create_modeling_pipeline(preprocessor, lr_model)

# Evaluate 5-fold CV on train set
cv_results = evaluate_cv_pipeline(baseline_pipeline, X_train, y_train, model_name="baseline_logistic_regression")

print("=== LOGISTIC REGRESSION BASELINE 5-FOLD CV METRICS ===")
for m, val in cv_results['metrics'].items():
    print(f"{m.upper():<10}: {val['mean']:.4f} +/- {val['std']:.4f}")


Saved cross-validation metrics: C:\Users\JINIL\Desktop\Sem5_Innovative\Phish_Guard\results\metrics\baseline_logistic_regression_cv_metrics.json
=== LOGISTIC REGRESSION BASELINE 5-FOLD CV METRICS ===
ACCURACY  : 0.9950 +/- 0.0003
PRECISION : 0.9985 +/- 0.0013
RECALL    : 0.9899 +/- 0.0019
F1        : 0.9942 +/- 0.0004
ROC_AUC   : 0.9986 +/- 0.0007
